building three real-world multi-agent AI systems using LangGraph and LangChain — covering Sequential Coordination (agents chaining outputs), Intent-Based Routing (LLM decides where to send a request), and Parallel Execution (multiple agents running concurrently).

Setup: Packages, Imports, LLM Initialization

In [3]:
from typing import TypedDict, Literal
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, START, END

In [4]:
# ── 0.3 Initialize the LLM ──────────────────────────────────
# Connects to your locally running Ollama server at localhost:11434
# Change model name here if you pulled a different one (e.g. "llama3.1:8b")
llm = ChatOllama(
    model="qwen2.5:3b",
    temperature=0.7,        # creativity level: 0 = deterministic, 1 = creative
    base_url="http://localhost:11434",  # default Ollama address
)

# ── 0.4 Sanity check ────────────────────────────────────────
response = llm.invoke([HumanMessage(content="Reply with one word: Ready")])
print("✅ Ollama connected:", response.content)

✅ Ollama connected: Ready


Sequential Agent Coordination:

In [5]:
# ============================================================
# SECTION 2 — Pattern 1: Sequential Agent Coordination
# Project : Job Application Assistant
# Flow    : resume_analyzer → cover_letter_writer → email_formatter
# ============================================================

# ── 2.1 STATE ───────────────────────────────────────────────
class JobApplicationState(TypedDict):
    job_description: str    # input from user
    resume_analysis: str    # output of Agent 1
    cover_letter: str       # output of Agent 2
    final_email: str        # output of Agent 3

# ── 2.2 NODES (AGENTS) ──────────────────────────────────────

def resume_analyzer_node(state: JobApplicationState) -> dict:
    """Agent 1 — Reads the job description, extracts key requirements."""
    print(" Agent 1: Analyzing job description...")
    response = llm.invoke([HumanMessage(content=f"""
        Analyze this job description and extract:
        1. Top 5 required skills
        2. Key responsibilities (3 bullet points)
        3. Ideal candidate profile (2 sentences)

        Job Description: {state['job_description']}
    """)])
    return {"resume_analysis": response.content}


def cover_letter_writer_node(state: JobApplicationState) -> dict:
    """Agent 2 — Uses Agent 1's analysis to write a cover letter."""
    print(" Agent 2: Writing cover letter...")
    response = llm.invoke([HumanMessage(content=f"""
        Write a professional 3-paragraph cover letter based on this job analysis.
        Highlight the key skills, show enthusiasm, keep it concise.

        Job Analysis   : {state['resume_analysis']}
        Job Description: {state['job_description']}
    """)])
    return {"cover_letter": response.content}


def email_formatter_node(state: JobApplicationState) -> dict:
    """Agent 3 — Takes the cover letter and wraps it into a ready-to-send email."""
    print("Agent 3: Formatting final email...")
    response = llm.invoke([HumanMessage(content=f"""
        Format this cover letter into a complete ready-to-send email.
        Add a subject line, professional greeting, and sign-off.

        Cover Letter: {state['cover_letter']}
    """)])
    return {"final_email": response.content}

# ── 2.3 BUILD GRAPH ─────────────────────────────────────────
# Sequential chain: each node's output becomes the next node's input via state

builder = StateGraph(JobApplicationState)

builder.add_node("resume_analyzer",    resume_analyzer_node)
builder.add_node("cover_letter_writer", cover_letter_writer_node)
builder.add_node("email_formatter",    email_formatter_node)

builder.add_edge(START,                 "resume_analyzer")
builder.add_edge("resume_analyzer",     "cover_letter_writer")
builder.add_edge("cover_letter_writer", "email_formatter")
builder.add_edge("email_formatter",     END)

job_app_graph = builder.compile()

# ── 2.4 RUN ─────────────────────────────────────────────────
job_description = """
Senior Python Developer at TechCorp
Requirements: 5+ years Python, FastAPI, PostgreSQL, AWS.
You will build scalable microservices and mentor junior developers.
Strong communication and problem-solving skills required.
"""

result = job_app_graph.invoke({"job_description": job_description})

print("\n" + "="*60)
print("RESUME ANALYSIS:\n", result["resume_analysis"])
print("\n COVER LETTER:\n",   result["cover_letter"])
print("\n FINAL EMAIL:\n",    result["final_email"])

 Agent 1: Analyzing job description...
 Agent 2: Writing cover letter...
Agent 3: Formatting final email...

RESUME ANALYSIS:
 Based on the provided job description for a Senior Python Developer position at TechCorp, here is an analysis of the top required skills, key responsibilities, and ideal candidate profile:

### Top 5 Required Skills:
1. **Python Proficiency**: Expertise in building scalable microservices using Python.
2. **FastAPI Framework**: Strong understanding and experience with FastAPI for developing web applications.
3. **PostgreSQL Database Integration**: Experience integrating and managing PostgreSQL databases.
4. **AWS Services Management**: Knowledge of AWS services, particularly their use in a tech environment.
5. **Mentoring Skills**: Ability to mentor junior developers effectively.

### Key Responsibilities:
1. **Developing Scalable Microservices**: Building and maintaining microservices architectures that can handle high loads efficiently.
2. **Managing Databases

Intent-Based Routing:

In [6]:
# ============================================================
# SECTION 3 - Pattern 2: Intent-Based Routing
# Project  : Customer Support Router
# Flow     : router_node -> billing / technical / general
# Ref      : https://docs.langchain.com/oss/python/langgraph/graph-api
# ============================================================

# -- 3.1 STATE -----------------------------------------------
class SupportState(TypedDict):
    user_query: str
    intent: str       # "billing" | "technical" | "general"
    response: str

# -- 3.2 ROUTER NODE -----------------------------------------
# Classifies the user query into one of three intents.
# temperature=0 ensures consistent, deterministic classification.

router_llm = ChatOllama(model="llama3.2", temperature=0)

def router_node(state: SupportState) -> dict:
    result = router_llm.invoke([HumanMessage(content=f"""
        Classify the query into exactly one category.
        Reply with only the category word, nothing else.

        Categories:
        - billing   (payments, charges, invoices, refunds)
        - technical (bugs, errors, crashes, not working)
        - general   (everything else)

        Query: {state['user_query']}
    """)])

    intent = result.content.strip().lower()
    if intent not in ["billing", "technical", "general"]:
        intent = "general"

    return {"intent": intent}

# -- 3.3 ROUTING FUNCTION ------------------------------------
# This is NOT a node. LangGraph calls this after router_node
# to decide which node to execute next.
# Ref: https://reference.langchain.com/python/langgraph/graph/state/StateGraph/add_conditional_edges

def route_by_intent(state: SupportState) -> Literal["billing", "technical", "general"]:
    return state["intent"]

# -- 3.4 SPECIALIST NODES ------------------------------------

def billing_agent(state: SupportState) -> dict:
    result = llm.invoke([HumanMessage(content=f"""
        You are a billing specialist.
        Respond clearly to this billing query.

        Query: {state['user_query']}
    """)])
    return {"response": result.content}


def technical_agent(state: SupportState) -> dict:
    result = llm.invoke([HumanMessage(content=f"""
        You are a technical support specialist.
        Provide step-by-step help for this issue.

        Query: {state['user_query']}
    """)])
    return {"response": result.content}


def general_agent(state: SupportState) -> dict:
    result = llm.invoke([HumanMessage(content=f"""
        You are a customer support agent.
        Answer this general inquiry helpfully and concisely.

        Query: {state['user_query']}
    """)])
    return {"response": result.content}

# -- 3.5 BUILD GRAPH -----------------------------------------

builder = StateGraph(SupportState)

builder.add_node("router",    router_node)
builder.add_node("billing",   billing_agent)
builder.add_node("technical", technical_agent)
builder.add_node("general",   general_agent)

builder.add_edge(START, "router")

builder.add_conditional_edges(
    "router",
    route_by_intent,
    {
        "billing":   "billing",
        "technical": "technical",
        "general":   "general",
    }
)

builder.add_edge("billing",   END)
builder.add_edge("technical", END)
builder.add_edge("general",   END)

support_graph = builder.compile()

# -- 3.6 RUN -------------------------------------------------

queries = [
    "I was charged twice for my subscription last month.",
    "The app crashes every time I upload a file.",
    "What are your business hours?",
]

for query in queries:
    result = support_graph.invoke({"user_query": query})
    print("Query  :", query)
    print("Intent :", result["intent"])
    print("Response:", result["response"])
    print("-" * 60)

Query  : I was charged twice for my subscription last month.
Intent : billing
Response: I apologize to hear that you were charged twice for your subscription last month. This is definitely not the expected service and could be an error on our end. 

Could you please provide me with more details about the services involved? Additionally, could you check if there are any additional charges or subscriptions under your account that might have been missed? Once I gather this information, we can proceed to rectify the issue.

Thank you for bringing this to my attention.
------------------------------------------------------------
Query  : The app crashes every time I upload a file.
Intent : technical
Response: Sure, I can help you troubleshoot the issue with your app crashing when uploading files. Here are some steps to diagnose and resolve the problem:

### Step 1: Identify Your App's Environment
Firstly, determine where and how your app is running (e.g., on a mobile device or in a browser)

In [ ]:
Parallel Agent Execution:

In [7]:
# ============================================================
# SECTION 4 - Pattern 3: Parallel Agent Execution
# Project  : Multilingual Translator
# Flow     : START -> spanish / french / japanese -> merge -> END
# Ref      : https://docs.langchain.com/oss/python/langgraph/graph-api
# ============================================================

# -- 4.1 STATE -----------------------------------------------
# All three translation fields are independent.
# Each translator node writes to its own field only.

class TranslationState(TypedDict):
    original_text: str
    spanish:       str
    french:        str
    japanese:      str
    final_report:  str

# -- 4.2 PARALLEL TRANSLATOR NODES ---------------------------
# These three nodes do not depend on each other.
# LangGraph fans out to all three simultaneously from START.

def translate_spanish(state: TranslationState) -> dict:
    result = llm.invoke([HumanMessage(content=f"""
        Translate the following text to Spanish.
        Return only the translation, nothing else.

        Text: {state['original_text']}
    """)])
    return {"spanish": result.content}


def translate_french(state: TranslationState) -> dict:
    result = llm.invoke([HumanMessage(content=f"""
        Translate the following text to French.
        Return only the translation, nothing else.

        Text: {state['original_text']}
    """)])
    return {"french": result.content}


def translate_japanese(state: TranslationState) -> dict:
    result = llm.invoke([HumanMessage(content=f"""
        Translate the following text to Japanese.
        Return only the translation, nothing else.

        Text: {state['original_text']}
    """)])
    return {"japanese": result.content}



def merge_results(state: TranslationState) -> dict:
    report = f"""
Original  : {state['original_text']}
Spanish   : {state['spanish']}
French    : {state['french']}
Japanese  : {state['japanese']}
    """
    return {"final_report": report}

# -- 4.4 BUILD GRAPH -----------------------------------------

builder = StateGraph(TranslationState)

builder.add_node("translate_spanish",  translate_spanish)
builder.add_node("translate_french",   translate_french)
builder.add_node("translate_japanese", translate_japanese)
builder.add_node("merge_results",      merge_results)

# Fan-out: START triggers all three nodes simultaneously
builder.add_edge(START, "translate_spanish")
builder.add_edge(START, "translate_french")
builder.add_edge(START, "translate_japanese")

# Fan-in: merge_results waits for all three to finish
builder.add_edge("translate_spanish",  "merge_results")
builder.add_edge("translate_french",   "merge_results")
builder.add_edge("translate_japanese", "merge_results")

builder.add_edge("merge_results", END)

translation_graph = builder.compile()

# -- 4.5 RUN -------------------------------------------------

result = translation_graph.invoke({
    "original_text": "Artificial intelligence is transforming the way we work and live."
})

print(result["final_report"])


Original  : Artificial intelligence is transforming the way we work and live.
Spanish   : La inteligencia artificial está transformando la manera en que trabajamos y vivimos.
French    : L'intelligence artificielle transforme la manière dont nous travaillons et vivons.
Japanese  : AIは私たちが働く方法と生活する方法を変革しています。
    
